In [1]:
# import necessary libraries
import sagemaker
from sagemaker.xgboost.estimator import XGBoost
from sagemaker.session import Session
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import (
    IntegerParameter,
    CategoricalParameter,
    ContinuousParameter,
    HyperparameterTuner
)
from sagemaker import get_execution_role
import boto3

role = get_execution_role()
default_bucket = "churndemo15"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


#### Code Explanation:
**Importing Libraries:**
- `sagemaker` — AWS SDK for managing ML workflows on SageMaker.
- `TrainingInput` — Defines the S3 data channels for training and validation.
- `HyperparameterTuner`, `IntegerParameter`, `ContinuousParameter` — Used to define the hyperparameter search space.
- `get_execution_role()` — Retrieves the IAM role attached to the SageMaker notebook, granting it permissions to access S3, launch training jobs, etc.
- `default_bucket` — The S3 bucket name where training data is stored.

## XGBoost Hyper-paramter Tuning and Training jobs

In [2]:
# Creating Training and Validation data channels from s3 buckets (saved in 'churn_data_prep.ipynb')
s3_input_train = TrainingInput(s3_data=f"s3://{default_bucket}/train.csv", content_type="csv")
s3_input_validation = TrainingInput(s3_data=f"s3://{default_bucket}/validation.csv", content_type="csv")

#### Code Explanation:
**Setting Up Data Channels:**
- `TrainingInput` tells SageMaker where to find the training and validation CSV files in S3.
- `content_type="csv"` specifies the file format.
- These channels (`train` and `validation`) are passed to the tuner so each training job automatically loads the correct data.

Following hyper-parameters are fixed 

- `metric` (default value for binary classification) error rate  = #(wrong_cases)/#(total_cases) at threshold of 0.5.
- `objective`  logistic regression for binary classification, output probability 
- `num_round` controls the number of boosting rounds. This is essentially the subsequent models that are trained using the residuals of previous iterations. Again, more rounds should produce a better fit on the training data, but can be computationally expensive or lead to overfitting.
- `rate_drop` The dropout rate that specifies the fraction of previous trees to drop during the dropout.



In [3]:
fixed_hyperparameters = {
    "eval_metric": "auc",
    "objective":"binary:logistic",
    "num_round":"100",
    "rate_drop":"0.3",
}

#### Code Explanation:
**Fixed Hyperparameters:**
These values remain constant across all tuning jobs:
- **`eval_metric: auc`** — Area Under the ROC Curve; measures how well the model distinguishes between churned and non-churned customers. Higher is better.
- **`objective: binary:logistic`** — Tells XGBoost this is a binary classification problem and to output probabilities.
- **`num_round: 100`** — The model trains for 100 boosting rounds.
- **`rate_drop: 0.3`** — Dropout rate for DART booster to prevent overfitting.

In [4]:
sess = sagemaker.Session()
container = sagemaker.image_uris.retrieve("xgboost", sess.boto_region_name, "1.5-1")

estimator = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    hyperparameters=fixed_hyperparameters,
    instance_type="ml.m4.xlarge",
    output_path="s3://{}/output".format(default_bucket),
    sagemaker_session=sess
)

#### Code Explanation:
**Creating the SageMaker Estimator:**
- `sagemaker.image_uris.retrieve("xgboost", ...)` fetches the official AWS XGBoost 1.5-1 Docker container URI for the current region.
- `sagemaker.estimator.Estimator(...)` wraps this container with our configuration:
  - `instance_type="ml.m4.xlarge"` — The EC2 instance used for training.
  - `output_path` — Where the trained model artifact (`.tar.gz`) is saved in S3.
  - `hyperparameters` — The fixed hyperparameters defined in the previous cell.

Following hyperparamters are varied for tuning:-

- `eta` controls how aggressive each round of boosting is. Larger values lead to more conservative boosting.
- `min_child_weight` Minimum sum of instance weight (hessian) needed in a child. If the tree partition step results in a leaf node with the sum of instance weight less than min_child_weight, the building process gives up further partitioning. The larger the tree, the more conservative it is.
- `alpha` L1 regularization term on weights. Increasing this value makes models more conservative.
- `max_depth` Maximum depth of a tree. Increasing this value makes the model more complex and likely to be overfit.

In [5]:
hyperparameter_ranges = {
    "eta": ContinuousParameter(0, 1),
    "min_child_weight": ContinuousParameter(1, 10),
    "alpha": ContinuousParameter(0, 2),
    "max_depth": IntegerParameter(1, 10),
}

#### Code Explanation:
**Hyperparameter Search Ranges:**
These are the parameters SageMaker will vary across training jobs to find the best combination:
- **`eta`** (0–1) — Learning rate; controls how much each tree contributes. Lower values = slower but more accurate learning.
- **`min_child_weight`** (1–10) — Minimum data points required in a leaf node. Higher values prevent overfitting.
- **`alpha`** (0–2) — L1 regularization; helps the model ignore irrelevant features.
- **`max_depth`** (1–10) — Maximum depth of each decision tree. Deeper trees capture more complexity but risk overfitting.

In [6]:
objective_metric_name = "validation:auc"

#### Code Explanation:
**Setting the Objective Metric:**
`validation:auc` tells the tuner which metric to optimize. SageMaker will monitor the AUC score on the **validation set** (not training set) for each job, and use it to decide which hyperparameter combination is best.

Using the validation set (not training set) prevents the tuner from selecting a model that simply memorizes the training data.

In [7]:
tuner = HyperparameterTuner(
    estimator=estimator,
    objective_metric_name="validation:auc",
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=[
        {
            "Name": "validation:auc",
            "Regex": "validation-auc:([0-9\\.]+)"
        }
    ],
    objective_type="Maximize",
    max_jobs=10,
    max_parallel_jobs=2
)

#### Code Explanation:
**Creating the HyperparameterTuner:**
- `objective_metric_name="validation:auc"` — The metric to maximize.
- `metric_definitions` — A regex pattern that tells SageMaker how to parse the AUC value from CloudWatch training logs.
- `objective_type="Maximize"` — Since higher AUC = better model, we maximize it.
- `max_jobs=10` — Total number of training jobs to run.
- `max_parallel_jobs=2` — How many jobs run simultaneously (balances speed vs cost).

SageMaker uses **Bayesian optimization** by default — it learns from previous jobs to make smarter choices for subsequent ones.

In [8]:
tuner.fit({
    "train":s3_input_train,
    "validation":s3_input_validation
    },include_cls_metadata=False)

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config
No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


............................................................................................!


#### Code Explanation:
**Launching the Tuning Job:**
`tuner.fit()` starts all hyperparameter tuning jobs on SageMaker. 
- The `train` and `validation` channels are passed so each training job receives the correct data.
- `include_cls_metadata=False` disables automatic metadata inference.

⏳ **This cell takes 15–20 minutes to complete.** SageMaker launches training jobs in parallel (2 at a time), evaluates each one, and uses the results to inform the next set of jobs.

In [9]:
tuning_job_result = boto3.client("sagemaker").describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name
)
job_count = tuning_job_result["TrainingJobStatusCounters"]["Completed"]
print("%d training jobs have completed" %job_count)

10 training jobs have completed


#### Code Explanation:
**Checking Completed Jobs:**
- `describe_hyper_parameter_tuning_job()` fetches the current status and results of the tuning job.
- `TrainingJobStatusCounters["Completed"]` gives the count of successfully finished training jobs.

This confirms how many jobs completed successfully before we proceed to analyze the results.

### Fetch Tuning results

In [10]:
import pandas as pd
tuner = sagemaker.HyperparameterTuningJobAnalytics(tuning_job_result["HyperParameterTuningJobName"])

full_df = tuner.dataframe()

objective = tuning_job_result["HyperParameterTuningJobConfig"]["HyperParameterTuningJobObjective"]
is_minimize = objective["Type"] != "Maximize"
objective_name = objective["MetricName"]

if len(full_df) > 0:
    df = full_df[full_df["FinalObjectiveValue"] > -float("inf")]
    if len(df) > 0:
        df = df.sort_values("FinalObjectiveValue", ascending=is_minimize)
        print("Number of training jobs with valid objective: %d" % len(df))
        print({"lowest": min(df["FinalObjectiveValue"]), "highest": max(df["FinalObjectiveValue"])})
        pd.set_option("display.max_colwidth", None)  # Don't truncate TrainingJobName
    else:
        print("No training jobs have reported valid results yet.")

df

Number of training jobs with valid objective: 10
{'lowest': 0.8189899921417236, 'highest': 0.850059986114502}


,alpha,eta,max_depth,min_child_weight,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
2,0.000000,0.166989,1.0,2.943739,sagemaker-xgboost-260415-0614-008-a9b7f9fa,Completed,0.85006,2026-04-15 06:20:26+00:00,2026-04-15 06:21:10+00:00,44.0
3,1.063324,0.108314,2.0,8.950011,sagemaker-xgboost-260415-0614-007-cf896bcd,Completed,0.84995,2026-04-15 06:20:14+00:00,2026-04-15 06:20:48+00:00,34.0
1,0.161188,0.114606,1.0,9.729530,sagemaker-xgboost-260415-0614-009-83ce8047,Completed,0.84825,2026-04-15 06:21:08+00:00,2026-04-15 06:21:42+00:00,34.0
4,0.848075,0.134363,4.0,9.182296,sagemaker-xgboost-260415-0614-006-d030203b,Completed,0.84816,2026-04-15 06:19:25+00:00,2026-04-15 06:20:08+00:00,43.0
0,0.583593,0.107468,1.0,6.501677,sagemaker-xgboost-260415-0614-010-b3578c2e,Completed,0.84766,2026-04-15 06:21:34+00:00,2026-04-15 06:22:17+00:00,43.0
7,1.075220,0.135869,4.0,6.524635,sagemaker-xgboost-260415-0614-003-07959ccf,Completed,0.84607,2026-04-15 06:17:54+00:00,2026-04-15 06:18:29+00:00,35.0
9,1.313865,0.226232,4.0,3.654075,sagemaker-xgboost-260415-0614-001-626f7959,Completed,0.84147,2026-04-15 06:15:52+00:00,2026-04-15 06:18:06+00:00,134.0
8,1.673617,0.210856,5.0,7.099861,sagemaker-xgboost-260415-0614-002-ca7bd674,Completed,0.83650,2026-04-15 06:15:23+00:00,2026-04-15 06:17:28+00:00,125.0
6,1.746415,0.205819,6.0,6.639341,sagemaker-xgboost-260415-0614-004-92f3c93c,Completed,0.83413,2026-04-15 06:18:28+00:00,2026-04-15 06:19:07+00:00,39.0
5,0.123499,0.439215,8.0,6.481000,sagemaker-xgboost-260415-0614-005-afc02e99,Completed,0.81899,2026-04-15 06:18:46+00:00,2026-04-15 06:19:20+00:00,34.0


#### Code Explanation:
**Analyzing Tuning Results:**
- `HyperparameterTuningJobAnalytics` fetches all job results as a DataFrame.
- We filter out jobs with invalid objective values (`-inf`).
- Results are sorted by `FinalObjectiveValue` (AUC score) to identify the best performing hyperparameter combination.
- The lowest and highest AUC scores are printed to show the range of model performance across all jobs.

In [11]:
best_hyperparameters = tuning_job_result["BestTrainingJob"]["TunedHyperParameters"]
best_hyperparameters

{'alpha': '0.0',
 'eta': '0.16698908802533913',
 'max_depth': '1',
 'min_child_weight': '2.9437393025712653'}

#### Code Explanation:
**Retrieving Best Hyperparameters:**
`tuning_job_result["BestTrainingJob"]["TunedHyperParameters"]` returns the exact hyperparameter values (`eta`, `max_depth`, `alpha`, `min_child_weight`) that produced the highest AUC on the validation set.

These are the values used when creating the final model for deployment.

The scatter plot shows that the points are distributed quite apart from each other. Hence, we have set the ranges well for hyperparamter optimization.


## Register the model

Go to Amazon Sagemaker console. In the Training -> Hyperparameter Tuning Jobs, select the hyperparamter tuning job with the corresponding name ,initiated in this notebook (or use Creation Time column).  There, in the hyperparamter tuning job, there will be tab showing the Best Trained Model summary and a button "Create Model", to  register the model container. We will later use this image to create a deployment endpoint for real-time prediction.